# 🇹🇼 台灣在地化中文語音轉文字轉錄器
## MediaTek Breeze-ASR-25 + 時間戳記對齊

---

### 📌 功能特色
- ✅ **一鍵執行**：無需重啟 kernel，首次即可成功
- 🇹🇼 **台灣本地化**：MediaTek Breeze-ASR-25 專為台灣華語優化
- 🚀 **GPU 自適應**：自動識別 T4/L4/A100/H100 並優化設定
- ⚡ **智能並行**：模型下載與檔案上傳同時進行，大幅縮短等待時間
- ⏱️ **時間戳記對齊**：每段文字標註起始與結束時間
- 📺 **SRT 字幕輸出**：可選輸出標準 SRT 字幕格式
- 📊 **超長音檔支援**：智能分段處理，支援數小時音檔
- 🗣️ **中英混用支援**：正確識別台灣常見的中英夾雜語句

---

In [ ]:
# =============================================================================
# 台灣在地化中文語音轉文字轉錄器 v4.2
# 使用 MediaTek Breeze-ASR-25 模型 + 時間戳記對齊
# =============================================================================
# 版本: 4.2.0 (2025-01)
# 新增: 時間戳記對齊功能、SRT 字幕輸出
# 優化: 模型下載與檔案上傳並行處理、改善進度顯示
# =============================================================================

import os
import sys
import subprocess
import warnings
import time
import gc
import threading
from typing import Optional, Tuple, Dict, Any, List
from concurrent.futures import ThreadPoolExecutor, Future
from dataclasses import dataclass
from IPython.display import display, clear_output
import ipywidgets as widgets

# 抑制警告
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# =============================================================================
# 第一部分：環境設定
# =============================================================================

def install_dependencies():
    """安裝依賴"""
    print("📦 正在檢查並安裝依賴套件...")
    start_time = time.time()

    packages = [
        ("transformers", "transformers>=4.36.0"),
        ("accelerate", "accelerate"),
        ("pydub", "pydub"),
        ("soundfile", "soundfile"),
        ("librosa", "librosa"),
    ]

    try:
        subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
        print("   ✓ ffmpeg 已安裝")
    except:
        print("   📥 安裝 ffmpeg...")
        subprocess.run(["apt-get", "-y", "update", "-qq"], capture_output=True)
        subprocess.run(["apt-get", "-y", "install", "ffmpeg", "-qq"], capture_output=True)

    for module_name, pkg_name in packages:
        try:
            __import__(module_name)
            print(f"   ✓ {module_name} 已安裝")
        except ImportError:
            print(f"   📥 安裝 {pkg_name}...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg_name],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )

    print(f"\n✅ 依賴安裝完成！耗時 {time.time() - start_time:.1f} 秒")

install_dependencies()

# =============================================================================
# 第二部分：匯入函式庫
# =============================================================================

import tempfile
import torch
import numpy as np
import soundfile as sf
from pydub import AudioSegment
from pydub.utils import mediainfo
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from google.colab import files

print("✅ 所有函式庫載入成功！")

# =============================================================================
# 第三部分：資料結構定義
# =============================================================================

@dataclass
class TimestampedSegment:
    """帶時間戳記的轉錄片段"""
    start_time: float  # 秒
    end_time: float    # 秒
    text: str

    def format_time(self, seconds: float, srt_format: bool = False) -> str:
        """格式化時間"""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = seconds % 60

        if srt_format:
            # SRT 格式: HH:MM:SS,mmm
            return f"{hours:02d}:{minutes:02d}:{secs:06.3f}".replace('.', ',')
        else:
            # 一般格式: HH:MM:SS
            return f"{hours:02d}:{minutes:02d}:{int(secs):02d}"

    def to_timestamp_line(self) -> str:
        """輸出時間戳記格式行"""
        start = self.format_time(self.start_time)
        end = self.format_time(self.end_time)
        return f"[{start} - {end}] {self.text}"

    def to_srt_block(self, index: int) -> str:
        """輸出 SRT 格式區塊"""
        start = self.format_time(self.start_time, srt_format=True)
        end = self.format_time(self.end_time, srt_format=True)
        return f"{index}\n{start} --> {end}\n{self.text}\n"

# =============================================================================
# 第四部分：GPU 偵測
# =============================================================================

class GPUOptimizer:
    GPU_CONFIGS = {
        "T4": {"torch_dtype": torch.float16, "chunk_seconds": 25, "desc": "Tesla T4 - 15GB"},
        "L4": {"torch_dtype": torch.float16, "chunk_seconds": 28, "desc": "L4 - 22.5GB"},
        "A100": {"torch_dtype": torch.float16, "chunk_seconds": 30, "desc": "A100 - 40GB"},
        "H100": {"torch_dtype": torch.bfloat16, "chunk_seconds": 30, "desc": "H100 - 80GB"},
        "V100": {"torch_dtype": torch.float16, "chunk_seconds": 25, "desc": "V100 - 16GB"},
        "CPU": {"torch_dtype": torch.float32, "chunk_seconds": 30, "desc": "CPU (較慢)"},
    }

    @staticmethod
    def detect() -> Tuple[str, Dict]:
        if not torch.cuda.is_available():
            print("⚠️ 未偵測到 GPU，使用 CPU 模式")
            return "CPU", GPUOptimizer.GPU_CONFIGS["CPU"]

        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)

        print(f"🔍 GPU: {gpu_name} ({vram:.1f} GB)")

        gpu_type = "T4"
        for key in ["H100", "A100", "L4", "V100", "T4"]:
            if key in gpu_name.upper():
                gpu_type = key
                break

        if vram >= 70: gpu_type = "H100"
        elif vram >= 35: gpu_type = "A100"
        elif vram >= 20: gpu_type = "L4"

        config = GPUOptimizer.GPU_CONFIGS[gpu_type]
        print(f"✅ 配置: {config['desc']}")
        return gpu_type, config

# =============================================================================
# 第五部分：音檔轉換
# =============================================================================

class AudioConverter:
    TARGET_SR = 16000

    @staticmethod
    def get_info(path: str) -> Dict:
        try:
            info = mediainfo(path)
            return {
                "sample_rate": int(info.get("sample_rate", 0)),
                "duration": float(info.get("duration", 0)),
            }
        except:
            return {}

    @staticmethod
    def convert(src_path: str) -> Tuple[str, Dict]:
        print("🎵 分析音檔...")
        info = AudioConverter.get_info(src_path)

        if info.get("sample_rate"):
            print(f"   原始: {info['sample_rate']} Hz, {info['duration']:.1f} 秒 ({info['duration']/60:.1f} 分鐘)")

        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            wav_path = tmp.name

        print("🔄 轉換中...")
        start = time.time()

        cmd = [
            "ffmpeg", "-y", "-threads", str(os.cpu_count() or 4),
            "-i", src_path, "-ar", str(AudioConverter.TARGET_SR),
            "-ac", "1", "-c:a", "pcm_s16le", "-loglevel", "error", wav_path
        ]

        try:
            subprocess.run(cmd, check=True, capture_output=True)
        except:
            audio = AudioSegment.from_file(src_path)
            audio = audio.set_frame_rate(AudioConverter.TARGET_SR).set_channels(1).set_sample_width(2)
            audio.export(wav_path, format="wav")

        print(f"✅ 音檔轉換完成！耗時 {time.time() - start:.1f} 秒")
        return wav_path, info

# =============================================================================
# 第六部分：Breeze-ASR-25 轉錄器 (含時間戳記)
# =============================================================================

class BreezeASR:
    MODEL_ID = "MediaTek-Research/Breeze-ASR-25"

    def __init__(self, device: str, torch_dtype: torch.dtype):
        self.device = device
        self.torch_dtype = torch_dtype
        self.model = None
        self.processor = None
        self._load_lock = threading.Lock()
        self._is_loaded = False
        self._load_status = "等待中"
        self._load_start_time = 0

    def load(self) -> 'BreezeASR':
        """載入模型（線程安全）"""
        with self._load_lock:
            if self._is_loaded:
                return self

            self._load_start_time = time.time()
            self._load_status = "下載 Processor..."
            self.processor = AutoProcessor.from_pretrained(self.MODEL_ID)

            self._load_status = "下載模型 (3.09 GB)..."
            self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
                self.MODEL_ID,
                torch_dtype=self.torch_dtype,
                low_cpu_mem_usage=True,
            )

            self._load_status = "載入至 GPU..."
            self.model.to(self.device)
            self.model.eval()

            self._is_loaded = True
            elapsed = time.time() - self._load_start_time
            self._load_status = f"完成 ({elapsed:.1f}秒)"
            return self

    def is_loaded(self) -> bool:
        return self._is_loaded

    def get_status(self) -> str:
        if self._is_loaded:
            return self._load_status
        elif self._load_start_time > 0:
            elapsed = time.time() - self._load_start_time
            return f"{self._load_status} ({elapsed:.0f}秒)"
        return self._load_status

    def transcribe_chunk(self, audio: np.ndarray) -> str:
        """轉錄單一分段"""
        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )
        input_features = inputs.input_features.to(self.device, dtype=self.torch_dtype)

        with torch.no_grad():
            predicted_ids = self.model.generate(
                input_features,
                max_new_tokens=440,
                num_beams=1,
                do_sample=False,
                return_timestamps=False,
            )

        text = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        del input_features, predicted_ids
        return text

    def transcribe(self, wav_path: str, chunk_seconds: int = 28) -> Tuple[List[TimestampedSegment], Dict]:
        """轉錄完整音檔，返回帶時間戳記的片段列表"""
        print("\n🎤 開始轉錄...")

        audio, sr = sf.read(wav_path)
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)

        total_duration = len(audio) / sr
        print(f"   時長: {total_duration:.1f} 秒 ({total_duration/60:.1f} 分鐘)")

        # 分段計算
        chunk_samples = chunk_seconds * sr
        overlap = int(0.5 * sr)  # 0.5 秒重疊
        step = chunk_samples - overlap
        num_chunks = max(1, int(np.ceil((len(audio) - overlap) / step)))

        print(f"   分段: {num_chunks} 段 (每段 {chunk_seconds} 秒)")

        segments: List[TimestampedSegment] = []
        errors = 0
        start_time = time.time()

        for i in range(num_chunks):
            start_sample = i * step
            end_sample = min(start_sample + chunk_samples, len(audio))
            chunk = audio[start_sample:end_sample]

            if len(chunk) < sr * 0.5:
                continue

            # 計算時間戳記
            chunk_start_time = start_sample / sr
            chunk_end_time = end_sample / sr

            progress = (i + 1) / num_chunks * 100
            print(f"\r   進度: {progress:5.1f}% ({i+1}/{num_chunks})", end="", flush=True)

            try:
                text = self.transcribe_chunk(chunk)
                if text.strip():
                    segment = TimestampedSegment(
                        start_time=chunk_start_time,
                        end_time=chunk_end_time,
                        text=text.strip()
                    )
                    segments.append(segment)
            except Exception as e:
                errors += 1
                if errors <= 3:
                    print(f"\n   ⚠️ 分段 {i+1} 錯誤: {str(e)[:100]}")

            if i % 10 == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

        print()  # 換行
        elapsed = time.time() - start_time

        total_chars = sum(len(seg.text) for seg in segments)

        info = {
            "duration": total_duration,
            "elapsed": elapsed,
            "speed": total_duration / elapsed if elapsed > 0 else 0,
            "chars": total_chars,
            "chunks": num_chunks,
            "segments": len(segments),
            "errors": errors,
            "success_rate": (num_chunks - errors) / num_chunks * 100 if num_chunks > 0 else 0
        }

        print(f"\n✅ 轉錄完成！")
        print(f"   時長: {total_duration:.1f} 秒 ({total_duration/60:.1f} 分鐘)")
        print(f"   片段: {len(segments)} 段")
        print(f"   字數: {total_chars} 字")

        return segments, info

# =============================================================================
# 第七部分：儲存結果 (含時間戳記)
# =============================================================================

def save_transcript_with_timestamps(
    segments: List[TimestampedSegment],
    src_name: str,
    output_srt: bool = True
) -> Tuple[Optional[str], Optional[str]]:
    """儲存帶時間戳記的轉錄結果"""

    if not segments:
        print("\n⚠️ 沒有成功轉錄任何內容，不建立檔案")
        return None, None

    base = os.path.splitext(os.path.basename(src_name))[0]

    # 儲存帶時間戳記的 TXT
    txt_path = f"/content/{base}_transcript.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        for segment in segments:
            f.write(segment.to_timestamp_line() + "\n")

    print(f"\n💾 已儲存 TXT: {txt_path}")

    # 儲存 SRT 字幕
    srt_path = None
    if output_srt:
        srt_path = f"/content/{base}_subtitle.srt"
        with open(srt_path, "w", encoding="utf-8") as f:
            for idx, segment in enumerate(segments, 1):
                f.write(segment.to_srt_block(idx) + "\n")
        print(f"💾 已儲存 SRT: {srt_path}")

    return txt_path, srt_path

# =============================================================================
# 第八部分：主程式
# =============================================================================

def main(output_srt: bool = True):
    """
    主程式

    Args:
        output_srt: 是否輸出 SRT 字幕檔 (預設 True)
    """
    print("\n" + "="*60)
    print("🇹🇼 台灣在地化中文語音轉文字轉錄器")
    print("   MediaTek Breeze-ASR-25 + 時間戳記對齊")
    print("="*60)

    wav_path = None
    asr = None
    executor = None

    try:
        # 步驟 1: GPU 偵測
        print("\n" + "-"*40)
        print("📊 步驟 1/5: GPU 偵測")
        print("-"*40)
        _, config = GPUOptimizer.detect()
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

        # 建立 ASR 實例
        asr = BreezeASR(device, config["torch_dtype"])

        # 步驟 2: 開始背景下載模型 + 等待使用者上傳
        print("\n" + "-"*40)
        print("📤 步驟 2/5: 上傳音檔 + 背景載入模型")
        print("-"*40)

        # 啟動背景模型下載
        executor = ThreadPoolExecutor(max_workers=1)
        model_future = executor.submit(asr.load)
        print("\n⚡ 背景開始下載模型 (3.09 GB)...")
        print("💡 請現在上傳您的音檔，兩者將並行處理！\n")

        # 等待使用者上傳檔案
        uploaded = files.upload()

        if not uploaded:
            print("❌ 未選擇檔案")
            return

        src_path = list(uploaded.keys())[0]
        print(f"\n✅ 檔案: {src_path}")
        print(f"   模型狀態: {asr.get_status()}")

        # 步驟 3: 轉換音檔
        print("\n" + "-"*40)
        print("🔄 步驟 3/5: 音檔轉換")
        print("-"*40)

        wav_path, _ = AudioConverter.convert(src_path)
        print(f"   模型狀態: {asr.get_status()}")

        # 步驟 4: 等待模型 + 轉錄
        print("\n" + "-"*40)
        print("🎤 步驟 4/5: 轉錄")
        print("-"*40)

        # 確保模型載入完成
        if not asr.is_loaded():
            print("⏳ 等待模型載入完成...")
            while not asr.is_loaded():
                print(f"\r   {asr.get_status()}        ", end="", flush=True)
                time.sleep(1)
            print()

        model_future.result()  # 確保沒有例外
        print(f"✅ 模型已就緒: {asr.get_status()}")

        segments, info = asr.transcribe(wav_path, config["chunk_seconds"])

        # 步驟 5: 儲存
        print("\n" + "-"*40)
        print("💾 步驟 5/5: 儲存")
        print("-"*40)

        txt_path, srt_path = save_transcript_with_timestamps(segments, src_path, output_srt)

        if txt_path and segments:
            # 顯示結果預覽
            print("\n" + "="*60)
            print("📝 轉錄結果預覽 (前 10 段)")
            print("="*60)

            for seg in segments[:10]:
                print(seg.to_timestamp_line())

            if len(segments) > 10:
                print(f"\n... (共 {len(segments)} 段，剩餘 {len(segments) - 10} 段請查看檔案) ...")

            print("="*60)

            print("\n📥 下載中...")
            files.download(txt_path)
            if srt_path:
                files.download(srt_path)

        print("\n✨ 完成！")

    except Exception as e:
        print(f"\n❌ 錯誤: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # 清理
        if wav_path and os.path.exists(wav_path):
            os.remove(wav_path)
            print("\n🧹 已清理暫存")

        if executor:
            executor.shutdown(wait=False)

        if asr and asr.model:
            del asr.model, asr.processor

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# =============================================================================
# 執行
# =============================================================================

if __name__ == "__main__":
    # 設定參數：
    # output_srt=True  -> 同時輸出 SRT 字幕檔
    # output_srt=False -> 僅輸出 TXT 時間戳記檔
    main(output_srt=True)